In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import *
from delta.tables import *

### Scenario:
You're given `sensor_readings_kv.txt` — raw output from an IoT sensor gateway. Each line is a semicolon-separated list of `key=value` pairs, but **not every sensor reports the same keys** — some send temperature only, some send humidity and pressure but no temperature, and the key order isn't fixed either. This is a genuinely common real-world shape for telemetry/IoT data, and it can't be handled with a fixed schema or `substring()` — you need to parse each line into key-value pairs dynamically, then pull out only the fields you care about, with missing ones becoming `null`.

**Problem:**

- Read the file as plain text (one row per line).
- Split each line on `;` to get an array of `key=value` strings.
- Turn that array into a **map** (`MapType`) — split each `key=value` pair on `=` and build a map per row (look into `str_to_map()`, which does exactly this in one step given the right delimiters).
- From the map, extract `sensor_id`, `timestamp`, `temperature`, `humidity`, and `pressure` as their own columns — use map key lookup (`col("map_col")["key"]` or `element_at()`). A row where a key is absent should produce `null` for that column, not an error.
- Cast `temperature`, `humidity`, and `pressure` to `double`.
- Order the output by `sensor_id` ascending, then `timestamp` ascending.

**Expected Output**

| sensor_id | timestamp | temperature | humidity | pressure |
| :--- | :--- | :--- | :--- | :--- |
| S1 | 2024-12-01 10:00:00 | 22.5 | 45.0 | null |
| S1 | 2024-12-01 10:05:00 | 22.7 | 44.0 | 1013.0 |
| S2 | 2024-12-01 10:00:00 | 19.8 | null | null |
| S3 | 2024-12-01 10:00:00 | null | 50.0 | 1012.0 |

In [0]:
readings_df = spark.read.text("/Workspace/Users/jeevan.busi8008@gmail.com/spark-practice/data/sensor_readings_kv.txt")
readings_df = readings_df.withColumn("split_values", expr("str_to_map(value, ';', '=')"))
output_df = readings_df.withColumns(
    {
        "sensor_id": col("split_values")["sensor_id"],
        "timestamp": col("split_values")["timestamp"],
        "temperature": col("split_values")["temperature"].cast(DoubleType()),
        "humidity": col("split_values")["humidity"].cast(DoubleType()),
        "pressure": col("split_values")["pressure"].cast(DoubleType())
    }
)
output_df = output_df.select("sensor_id", "timestamp", "temperature", "humidity", "pressure") \
    .orderBy(col("sensor_id").asc(), col("timestamp").asc())
output_df.show()

### Scenario:
You're given `user_events_polymorphic.json` — a **polymorphic event stream**, extremely common in real product analytics pipelines. Every record shares a few common fields (`event_id`, `event_type`, `user_id`, `timestamp`), but the rest of the payload depends entirely on `event_type`: `click` events have `element_id`, `purchase` events have `order_id`/`amount`, `signup` events have `referral_code`. When Spark reads this, it unions all possible fields across the whole file, so most rows will have several irrelevant `null` columns. A common real-world design is to keep the common fields as real columns, but collapse the event-specific fields into a single flexible `details` JSON string column — rather than a table with dozens of mostly-null columns as more event types get added over time.

**Problem — do both parts:**

**Part 1 — Normalize into a generic event table:**
- Read the JSON and check `.printSchema()` — notice how many nullable columns Spark inferred across the different event shapes.
- Build a `details` column: a JSON string containing **only the fields relevant to that row's event**, with no null keys included (look at `to_json(struct(...))` — by default it drops null fields from the struct automatically, which is exactly the behavior you want here).
- Final columns: `event_id`, `event_type`, `user_id`, `timestamp`, `details`.
- Order by `event_id` ascending.

**Part 2 — Purchase revenue per user:**
- From the same source, filter to `event_type = 'purchase'` only.
- Sum `amount` grouped by `user_id`.
- Order by `user_id` ascending.

**Expected Output — Part 1**

| event_id | event_type | user_id | timestamp | details |
| :--- | :--- | :--- | :--- | :--- |
| E1 | click | U1 | 2025-01-01 09:00:00 | {"element_id":"btn-buy"} |
| E2 | purchase | U1 | 2025-01-01 09:05:00 | {"order_id":"O100","amount":49.99} |
| E3 | signup | U2 | 2025-01-01 09:10:00 | {"referral_code":"REF123"} |
| E4 | click | U2 | 2025-01-01 09:15:00 | {"element_id":"btn-home"} |
| E5 | purchase | U2 | 2025-01-01 09:20:00 | {"order_id":"O101","amount":15.0} |

**Expected Output — Part 2**

| user_id | total_purchase_amount |
| :--- | :--- |
| U1 | 49.99 |
| U2 | 15.0 |

In [0]:
source_df = spark.read.format("json").load("/Workspace/Users/jeevan.busi8008@gmail.com/spark-practice/data/user_events_polymorphic.json")

"""
source_df.printSchema()
root
 |-- amount: double (nullable = true)
 |-- element_id: string (nullable = true)
 |-- event_id: string (nullable = true)
 |-- event_type: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- referral_code: string (nullable = true)
 |-- timestamp: string (nullable = true)
 |-- user_id: string (nullable = true)
"""
## part 1
event_df = source_df.withColumn(
    "details", to_json(
        struct(col("element_id"), col("referral_code"), col("order_id"), col("amount"))
    )
)
event_df.select("event_id", "event_type", "user_id", "timestamp", "details").show()

## part 2
user_purchase_df = source_df.filter(col("event_type") == "purchase") \
    .groupBy("user_id") \
        .agg(sum("amount").alias("total_purchase_amount")) \
            .orderBy(col("user_id").asc())
user_purchase_df.show()